# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library, based on the FAIR<sup>2</sup> dataset package.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Date published: {metadata.datePublished}")


## 2. Data Overview

Review available record sets, fields, and their `@id` values.

This helps you identify the keys you will use for extracting data.

In [ ]:
# Explore available record sets
print("Available record sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

# For each record set, show its fields and their @id
for rs in record_sets:
    print(f"\nRecord set '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields / columns:")
    for f in fields:
        name = f.get('name', '')
        desc = f.get('description', '')
        print(f"    - @id: {f['@id']} | name: {name} | desc: {desc}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For this dataset, we'll extract data from the main record set.
# (You can update 'main_record_set_id' if you want to analyze a different one.)

main_record_set_id = 'cr:MainRecordSet'  # You may need to adjust to the right @id from previous cell output

record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    print(f"Loading records for record set {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"  No records found for {rs_id}.")

# Select a record set with data (here, use the first with records)
selected_record_set = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set = rs_id
        break

if selected_record_set is not None:
    print(f"\nColumns for DataFrame from {selected_record_set}:\n{dataframes[selected_record_set].columns.tolist()}")
    display(dataframes[selected_record_set].head())
else:
    print("No record set with records was found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You can adapt the field `@id`s from the overview above for your own analyses.

In [ ]:
import numpy as np

# Specify a numeric field by @id. Replace with a suitable field @id as needed:
numeric_field_id = None
group_field_id = None

# Try to guess a numeric field (such as 'cr:abundance' or 'cr:coverage') from columns
if selected_record_set:
    df = dataframes[selected_record_set]
    col_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not col_candidates:
        # Try to guess columns by name heuristics
        col_candidates = [col for col in df.columns if any(x in col.lower() for x in ["intensity", "abundance", "count", "coverage", "mw", "weight", "peptide"])]
    if col_candidates:
        numeric_field_id = col_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")
    else:
        print("No numeric field detected.")
    # Group by a likely categorical field:
    group_candidates = [col for col in df.columns if ('sample' in col.lower() or 'type' in col.lower() or 'group' in col.lower())]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Selected group field: {group_field_id}")

    # Filter: keep only rows where the numeric value is finite and above a threshold
    if numeric_field_id is not None:
        # Try to cast to float for filtering
        col = df[numeric_field_id]
        col = pd.to_numeric(col, errors='coerce')
        threshold = np.nanmean(col) if np.nanmean(col) > 0 else 0
        filtered_df = df[col > threshold].copy()
        print(f"Filtering {numeric_field_id} > {threshold:.2f}, {len(filtered_df)} records remain.")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (col[col > threshold] - col.mean()) / col.std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group (if group_field_id is available):
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("\nGrouped mean values:")
            print(grouped_df.head())
    else:
        print("No numeric field for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, we'll plot the distribution of the numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(dataframes[selected_record_set][numeric_field_id], errors='coerce').dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field is available, show mean by group
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_record_set])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data or field selection for visualization.")

## 6. Conclusion

In this notebook, we've loaded and explored the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. We reviewed the dataset schema, explored available record sets and fields, extracted data into pandas DataFrames, and performed basic EDA and visualization by referencing data elements using their `@id`. You can further customize the analysis by specifying different record set and field IDs, or by expanding the EDA and visualization sections as needed for your research questions.
